![Logo](https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/shared_assets/logo.png)


**Developers:** Zoltan Barta  
**Date:** 2026-02-27  
**Version:** 2025-26/2

[<img src="https://colab.research.google.com/assets/colab-badge.svg">](https://colab.research.google.com/github/BartaZoltan/deep-reinforcement-learning-course/blob/main/notebooks/sessions/3_monte_carlo_methods/monte_carlo_methods_student.ipynb)

# Practice 3: Monte Carlo Methods

## Summary

We start this session with Monte Carlo state-value estimation on FrozenLake. The goal is to implement and compare First-Visit and Every-Visit MC prediction, then observe how stochasticity and sample size affect the learned values.


## Introduction

In the previous practice we solved reinforcement learning problems with Dynamic Programming, mainly Value Iteration and Policy Iteration. Those methods are elegant and theoretically clean, but they rely on a key ingredient: the transition model $p(s', r \mid s, a)$.
$$
p(s', r \mid s, a)
$$

In modern reinforcement learning, this transition model is usually not available. From the agent's perspective, the environment is often a black box: the agent can take actions, observe next states and rewards, but it does not know the inner mechanics in advance.

Monte Carlo (MC) methods are one of the most direct ways to handle this setting. Instead of solving the environment analytically, we sample complete episodes, observe what total return they produce, and use those sampled returns to estimate value functions.

This is the natural idea whenever a system is too complex or too hidden to analyze directly. If we cannot write down the internal mechanics, we can still interact with the system, collect examples, and learn from the observed outcomes. In reinforcement learning, the agent does exactly this: it tries actions, observes next states and rewards, and gradually builds an estimate of which situations and decisions are promising.

For episodic problems, the return from time step $t$ is:
$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots
$$

The basic intuition is simple:
- let the agent interact with the environment,
- collect full trajectories until termination,
- compute the return observed after visited states or state-action pairs,
- average these returns over many episodes.

A useful mental picture is the following: the agent finishes an episode, looks at the total return it received, and then uses that sampled outcome to update the visited states or state-action pairs. If a certain decision repeatedly appears in high-return episodes, its estimated value will increase. If it usually appears in bad episodes, its value will stay lower. After many episodes, these averages begin to reflect the long-term quality of the policy.

With enough data, these averages become increasingly reliable. Under standard coverage assumptions, repeated sampling lets Monte Carlo estimates converge toward the true value function. Intuitively, if relevant states and state-action pairs are visited often enough, the empirical average return approaches the expected return.

<div style="display:flex; justify-content:center; gap:24px; align-items:flex-start; margin:12px 0;">
  <div style="text-align:center; width:340px;">
    <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/DP.png" style="width:320px; height:220px; object-fit:contain;" />
    <div style="margin-top:8px;"><b>Dynamic Programming (DP)</b></div>
    <div style="font-size:13px; color:#555;">
      Uses the full transition model of the environment to compute value estimates exactly.
    </div>
  </div>
  <div style="text-align:center; width:340px;">
    <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/MC.png" style="width:320px; height:220px; object-fit:contain;" />
    <div style="margin-top:8px;"><b>Monte Carlo (MC)</b></div>
    <div style="font-size:13px; color:#555;">
      Estimates values from sampled complete episodes without requiring transition probabilities.
    </div>
  </div>
</div>

<div style="text-align:center; margin-top:12px; font-size:13px; max-width:760px; margin-left:auto; margin-right:auto;">
  <b>Figure.</b> Dynamic Programming and Monte Carlo both aim to estimate value functions, but they rely on different assumptions. Dynamic Programming is model-based: it assumes that the agent has access to the transition dynamics of the environment and can compute expected updates directly. Monte Carlo methods are model-free: they do not require transition probabilities and instead learn from many sampled episodes by averaging observed returns. This makes Monte Carlo more broadly applicable in black-box environments, but usually more data-hungry and higher-variance.
  <br><br>
  <b>Source.</b> [add citation here]
</div>


In this notebook, we start with prediction and then move toward control:
- First-Visit MC: update a state once per episode (at its first occurrence),
- Every-Visit MC: update at every occurrence in the episode,
- later, use the same sample-based idea for policy improvement.

This is also closely related to the policy-iteration logic from the previous notebook:
- start from a policy,
- generate trajectories using that policy,
- evaluate the returns produced by that policy,
- then improve the policy using the new estimates.

One important limitation is that Monte Carlo waits until the end of the episode before assigning credit. This means the final return is pushed back to all visited decisions, which is useful but can blur the difference between truly decisive actions and weak intermediate moves.

A good example is chess. Suppose the agent wins a full game and receives a final reward of $+1$. A pure Monte Carlo update uses the final outcome of that complete episode, so every move in the game receives credit from the same winning return. That includes both excellent moves and obvious mistakes. In other words, the method can tell that the overall trajectory ended well, but it cannot immediately separate the brilliant move that created the win from the blunder that almost lost the game. This delayed and coarse credit assignment is one of the main reasons why we later move toward Temporal-Difference methods and bootstrapping.

Reference: Sutton & Barto, Chapter 5.

In this notebook, we start with prediction and then move toward control:
- First-Visit MC: update a state once per episode (at its first occurrence),
- Every-Visit MC: update at every occurrence in the episode,
- later, use the same sample-based idea for policy improvement.

This is also closely related to the policy-iteration logic from the previous notebook:
- start from a policy,
- generate trajectories using that policy,
- evaluate the returns produced by that policy,
- then improve the policy using the new estimates.

One important limitation is that Monte Carlo waits until the end of the episode before assigning credit. This means the final return is pushed back to all visited decisions, which is useful but can blur the difference between truly decisive actions and weak intermediate moves.

A good example is chess. Suppose the agent wins a full game and receives a final reward of $+1$. A pure Monte Carlo update uses the final outcome of that complete episode, so every move in the game receives credit from the same winning return. That includes both excellent moves and obvious mistakes. In other words, the method can tell that the overall trajectory ended well, but it cannot immediately separate the brilliant move that created the win from the blunder that almost lost the game. This delayed and coarse credit assignment is one of the main reasons why we later move toward Temporal-Difference methods and bootstrapping.

Reference: Sutton & Barto, Chapter 5.

### Intuition: why is it called *Monte Carlo*?

The name comes from the Monte Carlo casino in Monaco and highlights the role of randomness and repeated sampling.

A classic example is estimating $\pi$ with random points:
- draw many random points uniformly inside a square,
- count how many fall inside an inscribed circle,
- use the fraction of points inside the circle to estimate the area ratio.

Because
$$
\frac{\text{area of circle}}{\text{area of square}} = \frac{\pi r^2}{4r^2} = \frac{\pi}{4},
$$
we can estimate
$$
\pi \approx 4 \cdot \frac{\text{points inside the circle}}{\text{total sampled points}}.
$$

The same logic appears in reinforcement learning: instead of computing exact quantities from a known model, we approximate them from many sampled experiences.


### Breakdown of two trajectories using Monte Carlo method - Simplified

<div style="display:flex; gap:16px; align-items:flex-start;">
  <div style="text-align:center;">
    <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/frozenlake_goal_run.gif" width="280" />
    <div>Successful trajectory</div>
  </div>
  <div style="text-align:center;">
    <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/frozenlake_hole_run.gif" width="280" />
    <div>Failure trajectory</div>
  </div>
</div>





<div style="overflow-x:auto; margin:10px 0 6px 0; white-space:nowrap;">
<table style="border-collapse:collapse; border:none; margin:0 auto; background:transparent; white-space:nowrap;">
  <tr>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_00_s0.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_01_a1_s4.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_02_a1_s8.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_03_a2_s9.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_04_a2_s10.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_05_a1_s14.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/goal_run/goal_06_a2_s15.png" width="84" /></td>
  </tr>
  <tr>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>S<sub>0</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>1</sub><br><span style="font-size:10px; font-weight:normal;">DOWN</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>4</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>1</sub><br><span style="font-size:10px; font-weight:normal;">DOWN</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>8</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>2</sub><br><span style="font-size:10px; font-weight:normal;">RIGHT</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>9</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>2</sub><br><span style="font-size:10px; font-weight:normal;">RIGHT</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>10</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>1</sub><br><span style="font-size:10px; font-weight:normal;">DOWN</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>14</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>2</sub><br><span style="font-size:10px; font-weight:normal;">RIGHT</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>15</sub> &rarr;<br><span style="font-size:10px; font-weight:normal;">Goal</span></b></td>
  </tr>
  <tr>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>1</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>2</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>3</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>4</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>5</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>6</sub> = 1</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
  </tr>
  <tr>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>0</sub>, a<sub>1</sub>) = 1/6</code><br><span style="font-size:9px;">= 0.167</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>4</sub>, a<sub>1</sub>) = 1/6</code><br><span style="font-size:9px;">= 0.167</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>8</sub>, a<sub>2</sub>) = 1/6</code><br><span style="font-size:9px;">= 0.167</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>9</sub>, a<sub>2</sub>) = 1/6</code><br><span style="font-size:9px;">= 0.167</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>10</sub>, a<sub>1</sub>) = 1/6</code><br><span style="font-size:9px;">= 0.167</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>14</sub>, a<sub>2</sub>) = 1/6</code><br><span style="font-size:9px;">= 0.167</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
  </tr>
</table>
</div>




### Breakdown of a failed episode trajectory - Simplified
<div style="overflow-x:auto; margin:10px 0 6px 0; white-space:nowrap;">
<table style="border-collapse:collapse; border:none; margin:0 auto; background:transparent; white-space:nowrap;">
  <tr>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/hole_00_s0.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/hole_01_a1_s4.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/hole_02_a1_s8.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/hole_03_a2_s9.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/hole_04_a2_s10.png" width="84" /></td>
    <td style="padding:0 5px; text-align:center; font-size:18px; font-weight:bold; border:none; background:transparent; white-space:nowrap;">&rarr;</td>
    <td style="padding:0 3px; text-align:center; border:none; background:transparent; white-space:nowrap;"><img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/manual_trajectories/hole_run/hole_05_a2_s11.png" width="84" /></td>
  </tr>
  <tr>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>S<sub>0</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>1</sub><br><span style="font-size:10px; font-weight:normal;">DOWN</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>4</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>1</sub><br><span style="font-size:10px; font-weight:normal;">DOWN</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>8</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>2</sub><br><span style="font-size:10px; font-weight:normal;">RIGHT</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>9</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>2</sub><br><span style="font-size:10px; font-weight:normal;">RIGHT</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>10</sub> &rarr;</b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:11px;"><b>a<sub>2</sub><br><span style="font-size:10px; font-weight:normal;">RIGHT</span></b></td>
    <td style="padding-top:3px; text-align:center; border:none; background:transparent; font-size:12px; white-space:nowrap;"><b>&rarr; S<sub>11</sub> &rarr;<br><span style="font-size:10px; font-weight:normal;">Hole</span></b></td>
  </tr>
  <tr>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>1</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>2</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>3</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>4</sub> = 0</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>r<sub>5</sub> = -1</code></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
  </tr>
  <tr>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>0</sub>, a<sub>1</sub>) = -1/5</code><br><span style="font-size:9px;">= -0.200</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>4</sub>, a<sub>1</sub>) = -1/5</code><br><span style="font-size:9px;">= -0.200</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>8</sub>, a<sub>2</sub>) = -1/5</code><br><span style="font-size:9px;">= -0.200</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>9</sub>, a<sub>2</sub>) = -1/5</code><br><span style="font-size:9px;">= -0.200</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
    <td style="padding-top:4px; text-align:center; border:none; background:transparent; font-size:10px;"><code>Q(S<sub>10</sub>, a<sub>2</sub>) = -1/5</code><br><span style="font-size:9px;">= -0.200</span></td>
    <td style="padding-top:6px; text-align:center; border:none; background:transparent;"></td>
  </tr>
</table>
</div>


<div style="display:flex; justify-content:center; margin:12px 0;">
  <div>

| $s_{0,0}$ | $s_{0,1}$ | $s_{0,2}$ | $s_{0,3}$ |
|---|---|---|---|
| `U: 0.00`<br>`D: -0.03`<br>`L: 0.00`<br>`R: 0.00` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: 0.00` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: 0.00` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: 0.00` |
| $s_{1,0}$ | $s_{1,1}$ | $s_{1,2}$ | $s_{1,3}$ |
| `U: 0.00`<br>`D: -0.03`<br>`L: 0.00`<br>`R: 0.00` | `Terminal state` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: 0.00` | `Terminal state` |
| $s_{2,0}$ | $s_{2,1}$ | $s_{2,2}$ | $s_{2,3}$ |
| `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: -0.03` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: -0.03` | `U: 0.00`<br>`D: 0.17`<br>`L: 0.00`<br>`R: -0.20` | `Terminal state` |
| $s_{3,0}$ | $s_{3,1}$ | $s_{3,2}$ | $s_{3,3}$ |
| `Terminal state` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: 0.00` | `U: 0.00`<br>`D: 0.00`<br>`L: 0.00`<br>`R: 0.17` | `Terminal state` |


  </div>
</div>



$$
\mathcal{S} = \{\, s_{r,c} \mid r,c \in \{0,1,2,3\} \,\}
$$

$$
S_t \in \mathcal{S}
$$

$$
S_t = s_{r,c}
$$

$S_t$ is the state at time step $t$, while $s_{r,c}$ is a fixed state in the grid and member of the state space.


### External Playground

For an interactive companion visualization, see **Farouq Aldori**'s Monte Carlo RL playground: [Monte Carlo RL Visualization](https://github.com/farouqaldori/monte-carlo-rl-visualization). It provides a useful sandbox for inspecting Monte Carlo rollouts and value updates alongside the ideas developed in this notebook. {cite}`aldori_monte_carlo_rl_visualization`


### Setup


In [ ]:
from __future__ import annotations

from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

try:
    import gymnasium as gym
except Exception:
    gym = None

from pathlib import Path
import importlib.util
import urllib.request


def _load_session3_utils():
    candidates = [
        Path('utils.py'),
        Path('notebooks/sessions/3_monte_carlo_methods/utils.py'),
        Path('/content/notebooks/sessions/3_monte_carlo_methods/utils.py'),
    ]

    utils_path = next((p for p in candidates if p.exists()), None)

    if utils_path is None:
        utils_path = Path('notebooks/sessions/3_monte_carlo_methods/utils.py')
        utils_path.parent.mkdir(parents=True, exist_ok=True)
        url = (
            'https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/'
            'notebooks/sessions/3_monte_carlo_methods/utils.py'
        )
        urllib.request.urlretrieve(url, utils_path)

    spec = importlib.util.spec_from_file_location('session3_utils', utils_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


s3u = _load_session3_utils()
SEED = 42
GLOBAL_RNG = s3u.set_seed(SEED)
CELL_OUTPUT_DIR = s3u.ensure_dir(Path('assets/cell_outputs'))


### FrozenLake environments

We start with FrozenLake 4x4 maps in two variants:
- deterministic (`is_slippery=False`) to debug estimator behavior,
- stochastic (`is_slippery=True`) to observe higher variance.


In [ ]:
if gym is None:
    raise RuntimeError('Gymnasium is required for this notebook block. Please install gymnasium.')

env_det = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False)
env_slip = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)

desc_det = env_det.unwrapped.desc
desc_slip = env_slip.unwrapped.desc

fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))
s3u.plot_frozenlake_map(desc_det, title='FrozenLake 4x4', ax=ax)
plt.tight_layout()
fig.savefig(CELL_OUTPUT_DIR / 'frozenlake_map.png', dpi=160, bbox_inches='tight')
plt.show()

print('nS=', env_det.observation_space.n, 'nA=', env_det.action_space.n)


### Task 1

**Generate one full episode from a fixed policy (10 min)**

Implement a helper function `generate_episode(env, policy_fn, max_steps=200)` that rolls out a single episode in the environment by repeatedly following the given policy.

Your function should:

- reset the environment with a random seed drawn from the global notebook seed,
- start from the initial state returned by `env.reset(...)`,
- at each step, call `policy_fn(state, nA)` to choose the next action,
- apply that action in the environment with `env.step(action)`,
- store the transition information as a tuple `(state, action, reward)`,
- stop when the episode terminates, truncates, or when `max_steps` is reached.

Return the full trajectory as a Python list of tuples:
`[(s_0, a_0, r_1), (s_1, a_1, r_2), ...]`

This function will be the basic data-collection building block for the Monte Carlo prediction methods in the next tasks, where we estimate values from complete episodes.


In [ ]:
def random_policy(state: int, number_of_actions: int) -> int:
    """Sample a uniformly random action.

    Parameters
    ----------
    state : int
        Current state index. Unused here, but kept for API consistency.
    number_of_actions : int
        Number of available actions.

    Returns
    -------
    int
        Sampled action index.
    """
    return int(GLOBAL_RNG.integers(0, number_of_actions))

In [ ]:
def generate_episode(env, policy_fn, max_steps: int = 200):
    """Roll out one complete episode.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    policy_fn : callable
        Policy function with signature ``policy_fn(state, nA)``.
    max_steps : int, default=200
        Maximum number of steps in the episode.

    Returns
    -------
    list[tuple[int, int, float]]
        Sequence of ``(state, action, reward)`` transitions.
    """
    trajectory = []
    ########################################################################










    ########################################################################
    return trajectory

In [ ]:
sample_ep = generate_episode(env_slip, random_policy, max_steps=50)
print('sample episode length:', len(sample_ep))
print('first 5 transitions:', sample_ep[:5])


## Monte Carlo Prediction - First-Visit & Every-Visit

Once we can generate complete episodes, we can use them to estimate the value of a given policy from experience.

For a fixed policy $\pi$, the goal of Monte Carlo prediction is to estimate the expected return:

$$
v_\pi(s) = \mathbb{E}_\pi [G_t \mid S_t = s]
$$

where the return from time step $t$ is

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots
$$

The key idea is simple: whenever a state appears in an episode, we look at the return observed after that visit, and over many episodes we average these returns.

There are two standard variants:

### First-Visit MC

In **First-Visit Monte Carlo**, we update a state only the first time it appears in a given episode.

- If a state is visited multiple times in the same trajectory, only the first occurrence is used.
- This gives at most one update per state per episode.

This method is slightly more conservative, since it avoids counting repeated visits within the same episode multiple times.

### Every-Visit MC

In **Every-Visit Monte Carlo**, we update a state every time it appears in the episode.

- If a state is visited multiple times, all occurrences are used.
- This may produce several updates for the same state within a single trajectory.

This method uses more data from each episode, but repeated visits from the same trajectory can be strongly correlated.

### Main Difference

The only difference is which visits are counted inside one episode:

- **First-Visit:** use only the first occurrence of each state
- **Every-Visit:** use all occurrences of each state

Both methods estimate the same value function and both converge under standard assumptions, but their finite-sample behavior can differ.



### Task 2

**Implement First-Visit and Every-Visit MC prediction (15 min)**

Using the episode generator from Task 1, implement two Monte Carlo prediction functions that estimate the action-value function of a fixed policy:

- `mc_first_visit_prediction(...)`
- `mc_every_visit_prediction(...)`

Both functions should estimate

$$
q_\pi(s,a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]
$$

from sampled episodes, where

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots
$$

Your implementation should follow this structure:

- initialize a table `Q` for all state-action pairs,
- maintain running sums and counts of observed returns for each state-action pair,
- generate `n_episodes` complete episodes using `generate_episode(...)`,
- process each episode backward so the return $G_t$ can be accumulated efficiently,
- update the estimate of each state-action pair by averaging all returns collected for that pair.

A subtle but important implementation detail is that it is much easier to compute the return by iterating **from the end of the episode back to the beginning**.

If we already know the return after time step $t+1$, then we can update the previous step with

$$
G_t = R_{t+1} + \gamma G_{t+1}
$$

So instead of recomputing the full discounted sum from scratch at every time step, we can reuse the return recursively while moving backward through the trajectory.

Implement the two variants as follows:

#### First-Visit MC

In `mc_first_visit_prediction(...)`, update a state-action pair only the **first time it appears in a given episode**.

A practical way to do this is:

- traverse the episode backward,
- keep track of which state-action pairs have already been updated for the current episode,
- skip repeated occurrences once that pair has already been counted.

This ensures that each state-action pair contributes at most one return sample per episode.

#### Every-Visit MC

In `mc_every_visit_prediction(...)`, update a state-action pair **every time it appears** in the episode.

In this version:

- traverse the episode backward,
- compute the return at each time step,
- and use every occurrence of every state-action pair without filtering repeated visits.

This means the same state-action pair may receive multiple updates from the same episode.

#### Goal

At the end, both functions should return a NumPy array `Q` containing the estimated action value for each state-action pair under the given policy.

These two functions estimate the same quantity, but they differ in how they count repeated state-action visits within a single episode. This makes them a good first comparison of two closely related Monte Carlo prediction methods.


<div style="display:flex; justify-content:center; margin:12px 0;">
  <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/FV_MCpred.png" style="width:760px; max-width:100%;" />
</div>

<div style="text-align:center; margin-top:10px; font-size:13px;">
  <b>Figure.</b> First-Visit Monte Carlo prediction pseudocode for estimating a fixed policy from complete episodes.
  <br>
  <i>Pseudocode adapted from Sutton & Barto {cite}`sutton2018` (Chapter 5).</i>
</div>

In [ ]:
def mc_prediction_update_from_episode(
    episode,
    old_Q: np.ndarray,
    returns_sum: np.ndarray,
    returns_count: np.ndarray,
    gamma: float,
    first_visit: bool = True,
):
    """Update action-value estimates from one episode.

    Parameters
    ----------
    episode : list[tuple[int, int, float]]
        Episode transitions as ``(state, action, reward)``.
    old_Q : np.ndarray
        Previous action-value table.
    returns_sum : np.ndarray
        Running sum of returns for each state-action pair.
    returns_count : np.ndarray
        Running count of returns for each state-action pair.
    gamma : float
        Discount factor.
    first_visit : bool, default=True
        Whether to use first-visit updates inside the episode.

    Returns
    -------
    np.ndarray
        Updated action-value table.
    """
    G = 0.0
    new_Q = old_Q.copy()
    first_visit_index = None
    if first_visit:
        first_visit_index = {}
        for t, (s, a, _) in enumerate(episode):
            sa = (s, a)
            if sa not in first_visit_index:
                first_visit_index[sa] = t

    ########################################################################










    ########################################################################
    return new_Q


In [ ]:
def mc_first_visit_prediction(env, policy_fn, n_episodes: int, gamma: float, max_steps: int = 200):
    """Estimate ``Q`` with first-visit Monte Carlo prediction.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    policy_fn : callable
        Policy used to generate episodes.
    n_episodes : int
        Number of episodes to sample.
    gamma : float
        Discount factor.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    np.ndarray
        Estimated action-value table.
    """
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    returns_sum = np.zeros((nS, nA), dtype=float)
    returns_count = np.zeros((nS, nA), dtype=float)

    for _ in range(n_episodes):
        episode = generate_episode(env, policy_fn, max_steps=max_steps)
        Q = mc_prediction_update_from_episode(
            episode,
            Q,
            returns_sum,
            returns_count,
            gamma,
            first_visit=True,
        )
    return Q    

In [ ]:

def mc_every_visit_prediction(env, policy_fn, n_episodes: int, gamma: float, max_steps: int = 200):
    """Estimate ``Q`` with every-visit Monte Carlo prediction.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    policy_fn : callable
        Policy used to generate episodes.
    n_episodes : int
        Number of episodes to sample.
    gamma : float
        Discount factor.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    np.ndarray
        Estimated action-value table.
    """
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA), dtype=float)
    returns_sum = np.zeros((nS, nA), dtype=float)
    returns_count = np.zeros((nS, nA), dtype=float)
    
    for _ in range(n_episodes):
        episode = generate_episode(env, policy_fn, max_steps=max_steps)
        Q = mc_prediction_update_from_episode(
            episode,
            Q,
            returns_sum,
            returns_count,
            gamma,
            first_visit=False,
        )

    return Q

### FrozenLake demonstration: First-Visit vs Every-Visit


In [ ]:
gamma = 0.99
n_episodes = 7000

Q_det_fv = mc_first_visit_prediction(env_det, random_policy, n_episodes=n_episodes, gamma=gamma)
Q_det_ev = mc_every_visit_prediction(env_det, random_policy, n_episodes=n_episodes, gamma=gamma)
Q_slip_fv = mc_first_visit_prediction(env_slip, random_policy, n_episodes=n_episodes, gamma=gamma)
Q_slip_ev = mc_every_visit_prediction(env_slip, random_policy, n_episodes=n_episodes, gamma=gamma)

s3u.plot_value_comparison(
    Q_det_fv,
    Q_det_ev,
    desc_det,
    title_a='First-Visit MC | deterministic',
    title_b='Every-Visit MC | deterministic',
    suptitle='FrozenLake 4x4 deterministic',
    save_path=CELL_OUTPUT_DIR / 'frozenlake_q_comparison_deterministic.png'
)

s3u.plot_value_comparison(
    Q_slip_fv,
    Q_slip_ev,
    desc_slip,
    title_a='First-Visit MC | slippery',
    title_b='Every-Visit MC | slippery',
    suptitle='FrozenLake 4x4 slippery',
    save_path=CELL_OUTPUT_DIR / 'frozenlake_q_comparison_slippery.png'
)


The deterministic case usually gives cleaner estimates with fewer episodes, while the slippery case requires more data because transition noise increases return variance. First-Visit and Every-Visit tend to converge toward similar values, but their finite-sample behavior can differ.


### Experiment: sample-size sensitivity


In [ ]:
episode_grid = [i for i in range(0, 10001, 500)]
start_state = 0

fv_curve = []
ev_curve = []

for n in episode_grid:
    Q_fv = mc_first_visit_prediction(env_slip, random_policy, n_episodes=n, gamma=0.99)
    Q_ev = mc_every_visit_prediction(env_slip, random_policy, n_episodes=n, gamma=0.99)
    fv_curve.append(np.mean(Q_fv[start_state]))
    ev_curve.append(np.mean(Q_ev[start_state]))

s3u.plot_curve_pair(
    episode_grid,
    fv_curve,
    ev_curve,
    label1='First-Visit',
    label2='Every-Visit',
    title='Start-state value vs number of episodes (slippery FrozenLake)',
    xlabel='Number of episodes',
    ylabel='Estimated V(s0)',
    save_path=CELL_OUTPUT_DIR / 'fv_vs_ev_start_state_curve.png'
)


The two curves stay fairly close to each other across the full training range, which shows that **First-Visit** and **Every-Visit** Monte Carlo are estimating the same underlying value and tend to produce similar results in practice.

At the same time, the curves are not perfectly smooth: they oscillate slightly as the number of episodes increases. This is expected, because Monte Carlo prediction is based on sampled trajectories, and in slippery FrozenLake those returns are noisy. Since this is still a **prediction** setting with a fixed random policy, the agent is not improving over time. What is stabilizing is the **value estimate**, not the episode behavior itself.


## From Prediction to Control

So far, we have only used Monte Carlo methods for **prediction**: we fixed a policy, sampled complete episodes, and estimated how good each state-action pair is under that policy.

The next step is to turn this into **control**.

The key idea is simple:

- **prediction** tells us how good the current policy is,
- **control** uses those value estimates to build a better policy.

In practice, this means:

1. start with some policy,
2. generate episodes with that policy,
3. estimate the action-value function $Q(s,a)$,
4. improve the policy by preferring actions with larger estimated values,
5. repeat.

This is the Monte Carlo version of the same evaluation-improvement loop we already saw in policy iteration.

A natural greedy improvement step would be:

$$
\pi_{\text{new}}(s) = \arg\max_a Q(s,a)
$$

However, if we become fully greedy too early, the agent may stop exploring and never revisit actions that were underestimated at the beginning.

To avoid this, we usually improve the policy in an **$\epsilon$-greedy** way:

- most of the time, choose the currently best-known action,
- but with small probability $\epsilon$, still explore.

This keeps the policy trainable while gradually pushing it toward better decisions.

So the transition from prediction to control is exactly this:

- estimate $Q(s,a)$ from complete episodes,
- then use those estimates to improve the policy,
- and keep repeating this loop until the policy stabilizes.



### Task 3

**Implement On-policy MC Control with $\epsilon$-greedy policy improvement (15-20 min)**

Using the Monte Carlo update logic from the previous task, complete the training loop inside:

- `mc_on_policy_control(...)`

This function learns an action-value function while gradually improving the policy over time. Unlike pure prediction, the policy is no longer fixed: after each episode, it should be updated using the latest `Q` estimates.

The surrounding setup is already provided for you:

- `Q`, `returns_sum`, and `returns_count` are already initialized,
- an `EpsilonGreedyActionSelector` is already created,
- a `TabularGreedyPolicy` is already created,
- and the policy object already knows how to:
  - sample actions,
  - store the current `Q` table,
  - return the final action-probability matrix,
  - return the final greedy action in each state.



Your code between should follow this structure:

- generate `n_episodes` complete episodes using the **current policy**,
- update `Q` from each episode using the shared Monte Carlo prediction update,
- use **first-visit** updates for state-action pairs inside each episode,
- overwrite the policy after each episode using the updated `Q`,
- after training, return the final `Q`, policy probabilities, greedy actions, and induced state values.

So the learning loop is:

1. generate an episode with the current policy,
2. evaluate that episode with a first-visit Monte Carlo update,
3. improve the policy using the new `Q`,
4. repeat.

#### Goal

At the end, the function should return:

- the learned action-value table `Q`,
- the final action-probability table for the policy,
- the greedy action in each state,
- and the state values induced by the final policy.

This task is the key step from **prediction** to **control**: instead of only estimating values for a fixed policy, we now repeatedly improve the policy using the estimates we learn from experience.


In [ ]:
class EpsilonGreedyActionSelector:
    """Simple epsilon-greedy action selector."""
    def __init__(self, epsilon: float):
        self.epsilon = float(epsilon)

    def select_action(self, preferred_action: int, nA: int) -> int:
        if float(GLOBAL_RNG.random()) < self.epsilon:
            return int(GLOBAL_RNG.integers(0, int(nA)))
        return int(preferred_action)

    def action_probs(self, preferred_action: int, nA: int) -> np.ndarray:
        probs = np.full(int(nA), self.epsilon / int(nA), dtype=float)
        probs[int(preferred_action)] += 1.0 - self.epsilon
        return probs

In [ ]:
class TabularGreedyPolicy:
    """Tabular greedy policy with epsilon-greedy sampling."""
    def __init__(self, nS: int, nA: int, action_selector: EpsilonGreedyActionSelector):
        self.nS = int(nS)
        self.nA = int(nA)
        self.Q = np.zeros((nS, nA), dtype=float)
        self.action_selector = action_selector
        
    def sample_greedy_action(self, state: int) -> int:
        q_row = self.Q[int(state)]
        best_value = np.max(q_row)
        best_actions = np.flatnonzero(q_row == best_value)
        return int(GLOBAL_RNG.choice(best_actions))

    def __call__(self, state: int, _nA: int):
        preferred_greedy_action = self.sample_greedy_action(state)
        return self.action_selector.select_action(preferred_greedy_action, self.nA)

    def update_q_table(self, new_Q: np.ndarray):
        self.Q = new_Q

    def get_q_table(self) -> np.ndarray:
        return self.Q

    @property
    def greedy_actions(self) -> np.ndarray:
        return np.argmax(self.Q, axis=1)

    def action_prob_matrix(self) -> np.ndarray:
        policy = np.zeros((self.nS, self.nA), dtype=float)
        for s in range(self.nS):
            policy[s] = self.action_selector.action_probs(self.greedy_actions[s], self.nA)
        return policy



In [ ]:
def mc_on_policy_control(
    env,
    n_episodes: int,
    gamma: float,
    epsilon: float,
    max_steps: int = 200,
):
    """Run first-visit on-policy Monte Carlo control.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    n_episodes : int
        Number of training episodes.
    gamma : float
        Discount factor.
    epsilon : float
        Exploration probability for the epsilon-greedy policy.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    dict[str, np.ndarray]
        Learned ``Q``, policy probabilities, greedy actions, and ``V``.
    """
    nS = env.observation_space.n
    nA = env.action_space.n

    action_selector = EpsilonGreedyActionSelector(epsilon)
    policy = TabularGreedyPolicy(nS, nA, action_selector)
    Q = policy.get_q_table().copy()

    returns_sum = np.zeros((nS, nA), dtype=float)
    returns_count = np.zeros((nS, nA), dtype=float)

    ########################################################################














    ########################################################################
    policy_matrix = policy.action_prob_matrix()
    greedy_actions = policy.greedy_actions.copy()
    V_pi = np.sum(policy_matrix * Q, axis=1)
    return {'Q': Q, 'policy': policy_matrix, 'greedy_actions': greedy_actions, 'V': V_pi}


### On-policy MC control demonstration (FrozenLake 4x4)


In [ ]:
def evaluate_greedy_success(env, greedy_actions: np.ndarray, n_eval_episodes: int, max_steps: int = 200):
    """Evaluate a deterministic greedy policy on FrozenLake.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    greedy_actions : np.ndarray
        Greedy action index for each state.
    n_eval_episodes : int
        Number of evaluation episodes.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    dict[str, float]
        Success rate and mean return.
    """
    success = 0
    avg_return = 0.0

    for _ in range(n_eval_episodes):
        obs, _ = env.reset(seed=int(GLOBAL_RNG.integers(0, 10_000_000)))
        ep_ret = 0.0
        for _ in range(max_steps):
            a = int(greedy_actions[int(obs)])
            obs, r, terminated, truncated, _ = env.step(a)
            ep_ret += float(r)
            if terminated or truncated:
                break

        avg_return += ep_ret
        # FrozenLake reward is 1 only when reaching goal.
        if ep_ret > 0.0:
            success += 1

    return {
        'success_rate': success / n_eval_episodes,
        'mean_return': avg_return / n_eval_episodes,
    }



In [ ]:
res_det = mc_on_policy_control(env_det, n_episodes=12_000, gamma=0.99, epsilon=0.10)
res_slip = mc_on_policy_control(env_slip, n_episodes=20_000, gamma=0.99, epsilon=0.10)

met_det = evaluate_greedy_success(env_det, res_det['greedy_actions'], n_eval_episodes=400)
met_slip = evaluate_greedy_success(env_slip, res_slip['greedy_actions'], n_eval_episodes=400)

print('Deterministic:', met_det)
print('Slippery:     ', met_slip)

s3u.plot_policy_comparison(
    res_det['greedy_actions'],
    res_slip['greedy_actions'],
    desc_det,
    title_a='Greedy policy | deterministic',
    title_b='Greedy policy | slippery',
    suptitle='On-policy MC control (epsilon=0.10)',
    save_path=CELL_OUTPUT_DIR / 'on_policy_control_policy_comparison.png'
)


The deterministic map converges to a cleaner policy with higher success rate, while the slippery map keeps a safer, noise-aware policy and typically needs more episodes for stable performance.


### Experiment: $\epsilon$ sensitivity (deterministic vs slippery)


In [ ]:
eps_grid = [0.01, 0.05, 0.10, 0.20, 0.50]
succ_det, succ_slip = [], []
ret_det, ret_slip = [], []

for eps in eps_grid:
    out_d = mc_on_policy_control(env_det, n_episodes=10_000, gamma=0.99, epsilon=eps)
    out_s = mc_on_policy_control(env_slip, n_episodes=20_000, gamma=0.99, epsilon=eps)

    met_d = evaluate_greedy_success(env_det, out_d['greedy_actions'], n_eval_episodes=300)
    met_s = evaluate_greedy_success(env_slip, out_s['greedy_actions'], n_eval_episodes=300)

    succ_det.append(100.0 * met_d['success_rate'])
    succ_slip.append(100.0 * met_s['success_rate'])
    ret_det.append(met_d['mean_return'])
    ret_slip.append(met_s['mean_return'])

s3u.plot_curve_pair(
    eps_grid,
    succ_det,
    succ_slip,
    label1='deterministic',
    label2='slippery',
    title='Success rate vs epsilon',
    xlabel='epsilon',
    ylabel='Success rate (%)',
    save_path=CELL_OUTPUT_DIR / 'epsilon_sensitivity_success_rate.png'
)

s3u.plot_curve_pair(
    eps_grid,
    ret_det,
    ret_slip,
    label1='deterministic',
    label2='slippery',
    title='Mean return vs epsilon',
    xlabel='epsilon',
    ylabel='Mean return',
    save_path=CELL_OUTPUT_DIR / 'epsilon_sensitivity_mean_return.png'
)


Smaller $\epsilon$ reduces exploration noise and can help deterministic performance, but too little exploration may slow correction of early bad estimates. In the slippery case, a moderate $\epsilon$ is usually more robust because stochastic transitions keep the value estimates noisy.


## Experiment: episode-budget sensitivity


In [ ]:
episode_grid_ctrl = [1000, 3000, 7000, 12000, 20000,50000]
succ_det_n, succ_slip_n = [], []

for n in episode_grid_ctrl:
    out_d = mc_on_policy_control(env_det, n_episodes=n, gamma=0.99, epsilon=0.10)
    out_s = mc_on_policy_control(env_slip, n_episodes=n, gamma=0.99, epsilon=0.10)

    met_d = evaluate_greedy_success(env_det, out_d['greedy_actions'], n_eval_episodes=300)
    met_s = evaluate_greedy_success(env_slip, out_s['greedy_actions'], n_eval_episodes=300)

    succ_det_n.append(100.0 * met_d['success_rate'])
    succ_slip_n.append(100.0 * met_s['success_rate'])

s3u.plot_curve_pair(
    episode_grid_ctrl,
    succ_det_n,
    succ_slip_n,
    label1='deterministic',
    label2='slippery',
    title='Success rate vs number of training episodes',
    xlabel='Training episodes',
    ylabel='Success rate (%)',
    save_path=CELL_OUTPUT_DIR / 'training_episodes_success_rate.png'
)


As the training budget grows, both environments improve, but the slippery case scales more slowly. This is the main transition point toward TD methods: Monte Carlo is conceptually simple, but it often needs many full episodes in noisy tasks.



## On-policy vs Off-policy

So far, the Monte Carlo methods in this notebook were **on-policy**.

This means that the same policy plays two roles at once:

- it **generates the episodes**,
- and it is also the policy we want to **evaluate or improve**.

In other words, the agent learns from trajectories produced by its **current own behavior**.

Formally, if the policy is $\pi$, then in the on-policy case we estimate values such as

$$
v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]
$$

or

$$
q_\pi(s,a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]
$$

using episodes that were also sampled under the same policy $\pi$.



<div style="display:flex; align-items:flex-start; gap:24px; margin:12px 0;">
  <div style="flex:0 0 280px; text-align:center;">
    <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/ON-offpo.drawio.png" style="width:280px; height:auto;" />
    <div style="margin-top:8px; font-size:13px;">
      <em>Conceptual comparison of on-policy and off-policy learning.</em>
    </div>
  </div>

  <div style="flex:1; font-size:14px; line-height:1.6;">
    <p style="margin-top:0;">
      <strong>On-policy learning.</strong>
    </p>
    <p>
      This is the simplest setup:
    </p>
    <ul style="margin-top:0;">
      <li>follow a policy,</li>
      <li>collect complete episodes,</li>
      <li>estimate returns from those episodes,</li>
      <li>improve the same policy.</li>
    </ul>
    <p>
      In other words, the same policy both generates the episodes and is also the policy we evaluate or improve.
      The agent therefore learns directly from its current own behavior.
    </p>
    <p>
      If the policy is <em>&pi;</em>, then the sampled trajectories also come from <em>&pi;</em>.
      So we estimate values such as <em>v</em><sub>&pi;</sub>(<em>s</em>) or
      <em>q</em><sub>&pi;</sub>(<em>s,a</em>) directly from episodes generated by that same policy.
    </p>
    <p>
      This is exactly what we did in on-policy Monte Carlo control with an <em>&epsilon;</em>-greedy policy:
      the agent mostly chooses the current best action, but still explores occasionally.
    </p>
    <p>
      <strong>Off-policy learning.</strong>
    </p>
    <p>
      In the off-policy setting, we separate these two roles.
    </p>
    <p style="margin-bottom:6px;">
      We now have two different policies:
    </p>
    <ul style="margin-top:0;">
      <li><strong>behavior policy</strong> <em>b</em>: the policy that actually generates the data</li>
      <li><strong>target policy</strong> <em>&pi;</em>: the policy whose value we want to estimate</li>
    </ul>
    <p>
      So the key question becomes:
      <br>
      <em>How can we estimate the value of <em>&pi;</em> using episodes generated by <em>b</em>?</em>
    </p>
    <p>
      This is useful because in practice we often want to:
    </p>
    <ul style="margin-top:0;">
      <li>keep exploring with a noisy or random policy,</li>
      <li>while evaluating a more greedy policy,</li>
      <li>or reuse data collected by some older policy.</li>
    </ul>
    <p style="margin-bottom:6px;">In this notebook, the typical setup is:</p>
    <ul style="margin-top:0;">
      <li><strong>behavior policy</strong>: <em>&epsilon;</em>-greedy</li>
      <li><strong>target policy</strong>: pure greedy and deterministic</li>
    </ul>
    <p>
      So the data is exploratory, but the value estimate is for a stricter target policy.
    </p>
  </div>
</div>





### Why a plain Monte Carlo average no longer works

If the episodes come from $b$ instead of $\pi$, then a direct sample average is no longer estimating the correct expectation under $\pi$.

The trajectories reflect the probabilities of the **behavior policy**, not the **target policy**.

That means we must correct for the mismatch between the two policies.



### Importance Sampling

The standard correction is called **importance sampling**.

The idea is simple:

- if a trajectory is more likely under the target policy, it should get a larger weight,
- if it is less likely under the target policy, it should get a smaller weight,
- if the target policy would never choose one of the observed actions, that trajectory gets weight zero from that point on.

The importance sampling ratio for one trajectory suffix is

$$
\rho_{t:T-1} = \prod_{k=t}^{T-1} \frac{\pi(A_k \mid S_k)}{b(A_k \mid S_k)}
$$

This ratio tells us how much more or less probable the observed action sequence is under the target policy than under the behavior policy.

We then reweight the return:

$$
v_\pi(s) = \mathbb{E}_b[\rho_{t:T-1} G_t \mid S_t = s]
$$

So we still use Monte Carlo returns, but now each return is scaled by an importance weight.



#### Ordinary Importance Sampling

- directly averages the weighted returns,
- unbiased in theory,
- but often has very high variance.

#### Weighted Importance Sampling

- normalizes by the total weight,
- usually more stable in practice,
- but biased in finite samples.

In most practical Monte Carlo experiments, **weighted importance sampling** is often easier to interpret because the curve is less noisy.




In [ ]:
def make_policy_from_greedy_actions(greedy_actions: np.ndarray, nA: int, epsilon: float):
    """Create a tabular policy from greedy action labels.

    Parameters
    ----------
    greedy_actions : np.ndarray
        Greedy action for each state.
    nA : int
        Number of actions.
    epsilon : float
        Exploration probability.

    Returns
    -------
    TabularGreedyPolicy
        Policy initialized around the provided greedy actions.
    """
    nS = len(greedy_actions)
    selector = EpsilonGreedyActionSelector(epsilon)
    policy = TabularGreedyPolicy(nS, nA, selector)

    Q_seed = np.zeros((nS, nA), dtype=float)
    for s, a_star in enumerate(greedy_actions):
        Q_seed[int(s), int(a_star)] = 1.0

    policy.update_q_table(Q_seed)
    return policy

In [ ]:

def policy_action_prob(policy: TabularGreedyPolicy, state: int, action: int) -> float:
    """Return the probability of one action under a policy.

    Parameters
    ----------
    policy : TabularGreedyPolicy
        Policy instance.
    state : int
        State index.
    action : int
        Action index.

    Returns
    -------
    float
        Action probability.
    """
    preferred_action = int(policy.greedy_actions[int(state)])
    probs = policy.action_selector.action_probs(preferred_action, policy.nA)
    return float(probs[int(action)])


In [ ]:
def generate_offpolicy_episode(
    env,
    target_policy: TabularGreedyPolicy,
    behavior_policy: TabularGreedyPolicy,
    max_steps: int = 200,
):
    """Generate one behavior-policy episode with stored policy probabilities.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    target_policy : TabularGreedyPolicy
        Policy being evaluated.
    behavior_policy : TabularGreedyPolicy
        Policy used to sample the episode.
    max_steps : int, default=200
        Maximum number of steps in the episode.

    Returns
    -------
    list[tuple[int, int, float, float, float]]
        Transitions with stored target and behavior probabilities.
    """
    base_episode = generate_episode(env, behavior_policy, max_steps=max_steps)
    traj = []

    for s, a, r in base_episode:
        pi_prob = policy_action_prob(target_policy, s, a)
        b_prob = policy_action_prob(behavior_policy, s, a)
        traj.append((int(s), int(a), float(r), float(pi_prob), float(b_prob)))

    return traj

### Task 4

**Implement Off-policy MC prediction with importance sampling (15-20 min)**

Using the policy and episode-generation helpers from the previous sections, implement an off-policy Monte Carlo prediction function:

- `off_policy_mc_prediction_start_state(...)`

The goal of this task is to estimate the **start-state value** of a fixed greedy target policy while the actual episodes are generated by a different exploratory behavior policy.

In other words:

- the **target policy** is the deterministic greedy policy we want to evaluate,
- the **behavior policy** is an $\epsilon$-greedy version of that policy, used to generate data.

So unlike the on-policy case, the trajectories are not sampled from the same policy whose value we want to estimate.

Your function should estimate the start-state value using importance sampling.

The return is still

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots
$$

but now it must be reweighted by the importance ratio

$$
\rho = \prod_{k=t}^{T-1} \frac{\pi(A_k \mid S_k)}{b(A_k \mid S_k)}
$$

where:

- $\pi$ is the target policy,
- $b$ is the behavior policy.

The surrounding setup is already provided for you:

- the target policy is constructed from the greedy action table with `epsilon=0.0`,
- the behavior policy is constructed from the same greedy action table with `epsilon=epsilon_b`,
- helper functions already exist for:
  - creating these policies,
  - generating an off-policy episode,
  - and retrieving $\pi(a \mid s)$ and $b(a \mid s)$ for each transition.

Your implementation should follow this structure:

- initialize lists for the running ordinary and weighted estimates,
- maintain a running numerator for ordinary importance sampling,
- maintain a cumulative weight and running estimate for weighted importance sampling,
- sample `n_episodes` off-policy episodes,
- traverse each episode backward to compute the discounted return,
- accumulate the importance ratio at the same time,
- update both estimators after each episode.

A subtle but important detail is that the return and the importance ratio should both be accumulated by moving **backward through the episode**.

This is convenient because:

- the return can be updated recursively as usual,
- and the importance ratio can be built step by step from the end of the trajectory.

If at any point the target policy would assign zero probability to an observed action, then:

- the ratio becomes zero,
- and that episode stops contributing to the off-policy estimate from that point on.

Implement the two estimates as follows:

#### Ordinary importance sampling

For the ordinary estimator:

- accumulate the weighted return $\rho G$,
- divide by the total number of sampled episodes.

This estimator is unbiased in theory, but it can have very high variance.

#### Weighted importance sampling

For the weighted estimator:

- keep a cumulative sum of the nonzero importance weights,
- update the estimate using the normalized weight contribution of each episode.

This version is typically more stable in practice, even though it is biased in finite samples.

#### Goal

At the end, the function should return:

- the full running curve for the ordinary importance sampling estimate,
- the full running curve for the weighted importance sampling estimate,
- and the fraction of episodes with nonzero importance ratio.

This task is the key introduction to **off-policy** Monte Carlo learning: instead of learning from episodes generated by the target policy itself, we now reuse exploratory data and correct it with importance sampling.

In [ ]:
def off_policy_mc_prediction_start_state(
    env,
    greedy_actions: np.ndarray,
    n_episodes: int,
    epsilon_b: float,
    gamma: float,
    max_steps: int = 200,
):
    """Estimate the start-state value with off-policy Monte Carlo.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    greedy_actions : np.ndarray
        Deterministic greedy target policy.
    n_episodes : int
        Number of sampled episodes.
    epsilon_b : float
        Behavior-policy exploration rate.
    gamma : float
        Discount factor.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    dict[str, np.ndarray | float]
        Ordinary IS curve, weighted IS curve, and nonzero-ratio fraction.
    """
    nA = env.action_space.n
    target_policy = make_policy_from_greedy_actions(greedy_actions, nA, epsilon=0.0)
    behavior_policy = make_policy_from_greedy_actions(greedy_actions, nA, epsilon=epsilon_b)

    ordinary = []
    weighted = []

    ordinary_num = 0.0
    weighted_c = 0.0
    weighted_v = 0.0
    nonzero_rho = 0

    for i in range(1, n_episodes + 1):
    ########################################################################




























    ########################################################################
    return {
        'ordinary': np.array(ordinary, dtype=float),
        'weighted': np.array(weighted, dtype=float),
        'nonzero_ratio_frac': nonzero_rho / n_episodes,
    }


### Off-policy MC prediction on slippery FrozenLake

We evaluate the start-state value of a deterministic greedy target policy using episodes generated by an $\epsilon$-greedy behavior policy.


In [ ]:
target_actions = res_slip['greedy_actions']

off = off_policy_mc_prediction_start_state(
    env=env_slip,
    greedy_actions=target_actions,
    n_episodes=8000,
    epsilon_b=0.20,
    gamma=0.99,
)

print(f"Fraction of episodes with non-zero importance ratio: {off['nonzero_ratio_frac']*100:.2f}%")

x = np.arange(1, len(off['ordinary']) + 1)
s3u.plot_curve_pair(
    x,
    off['ordinary'],
    off['weighted'],
    label1='Ordinary IS',
    label2='Weighted IS',
    title='Off-policy start-state value estimate over episodes',
    xlabel='Episode index',
    ylabel='Estimated V(s0)',
    save_path=CELL_OUTPUT_DIR / 'off_policy_is_curves.png'
)


The two curves show how the off-policy estimate of the start-state value evolves as more behavior-policy episodes are collected.

The **ordinary importance sampling** curve is usually much noisier. This is because some episodes receive very large importance weights, so a few trajectories can strongly influence the estimate. As a result, the ordinary estimator often fluctuates more and can be unstable in finite samples.

The **weighted importance sampling** curve is typically smoother and more stable. By normalizing with the cumulative sum of importance weights, it reduces the effect of extreme individual episodes. In practice, this often makes the weighted estimate easier to interpret, even though it is biased in finite samples.



Weighted importance sampling is usually more stable in finite samples, while ordinary importance sampling can show large variance when a few episodes get very high importance ratios.


### Experiment: behavior exploration level


In [ ]:
eps_b_grid = [0.05, 0.10, 0.20, 0.30]
final_ord = []
final_wis = []
nonzero_frac = []

for eps_b in eps_b_grid:
    out = off_policy_mc_prediction_start_state(
        env=env_slip,
        greedy_actions=target_actions,
        n_episodes=5000,
        epsilon_b=eps_b,
        gamma=0.99,
    )
    final_ord.append(float(out['ordinary'][-1]))
    final_wis.append(float(out['weighted'][-1]))
    nonzero_frac.append(100.0 * out['nonzero_ratio_frac'])

s3u.plot_bar_pair(
    [str(e) for e in eps_b_grid],
    final_ord,
    final_wis,
    name_a='Ordinary IS',
    name_b='Weighted IS',
    title='Final off-policy estimate vs behavior epsilon',
    ylabel='Final estimated V(s0)',
    save_path=CELL_OUTPUT_DIR / 'off_policy_behavior_epsilon_bar.png'
)

fig = plt.figure(figsize=(6.5, 3.8))
plt.plot(eps_b_grid, nonzero_frac, marker='o')
plt.title('Episodes with non-zero importance ratio')
plt.xlabel('Behavior epsilon')
plt.ylabel('Non-zero ratio episodes (%)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(CELL_OUTPUT_DIR / 'off_policy_nonzero_ratio.png', dpi=160, bbox_inches='tight')
plt.show()


This experiment shows how the amount of exploration in the **behavior policy** affects off-policy estimation.

As the behavior policy becomes more exploratory (larger $\epsilon_b$), it deviates more often from the deterministic greedy target policy. This changes two important things:

- it affects the final value estimate,
- and it affects how many sampled episodes still have a nonzero importance ratio.

In the bar chart, we compare the final **ordinary** and **weighted** importance sampling estimates for different values of $\epsilon_b$.

A smaller $\epsilon_b$ means the behavior policy stays closer to the target policy, so sampled episodes are more likely to match the target and contribute useful weight. A larger $\epsilon_b$ increases exploration, but it also increases mismatch between behavior and target, which usually makes the off-policy estimate noisier.

The second plot shows the fraction of episodes with **non-zero importance ratio**.

This is especially important here because the target policy is deterministic:

- if the behavior policy takes an action that the target policy would never take, the importance ratio becomes zero,
- so that episode stops contributing to the target estimate.

As a result, when $\epsilon_b$ increases, the fraction of useful episodes often decreases.


In [ ]:
def running_mean_episode_returns(env, policy_fn, n_episodes: int, max_steps: int = 200):
    """Compute raw and running-mean episode returns.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    policy_fn : callable
        Policy used to sample episodes.
    n_episodes : int
        Number of episodes.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Raw episode returns and running means.
    """
    returns = []
    running_mean = []
    total = 0.0

    for i in range(1, n_episodes + 1):
        episode = generate_episode(env, policy_fn, max_steps=max_steps)
        ep_return = float(sum(r for _, _, r in episode))
        total += ep_return
        returns.append(ep_return)
        running_mean.append(total / i)

    return np.array(returns, dtype=float), np.array(running_mean, dtype=float)


def on_policy_mc_prediction_start_state_curve(
    env,
    greedy_actions: np.ndarray,
    n_episodes: int,
    gamma: float,
    max_steps: int = 200,
):
    """Track the on-policy start-state estimate across episodes.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    greedy_actions : np.ndarray
        Deterministic target policy.
    n_episodes : int
        Number of sampled episodes.
    gamma : float
        Discount factor.
    max_steps : int, default=200
        Maximum number of steps per episode.

    Returns
    -------
    np.ndarray
        Sequence of start-state estimates.
    """
    nS = env.observation_space.n
    nA = env.action_space.n
    target_policy = make_policy_from_greedy_actions(greedy_actions, nA, epsilon=0.0)
    target_action_s0 = int(target_policy.greedy_actions[0])

    Q = np.zeros((nS, nA), dtype=float)
    returns_sum = np.zeros((nS, nA), dtype=float)
    returns_count = np.zeros((nS, nA), dtype=float)
    estimates = []

    for _ in range(n_episodes):
        episode = generate_episode(env, target_policy, max_steps=max_steps)
        Q = mc_prediction_update_from_episode(
            episode,
            Q,
            returns_sum,
            returns_count,
            gamma,
            first_visit=True,
        )
        estimates.append(float(Q[0, target_action_s0]))

    return np.array(estimates, dtype=float)


def evaluate_greedy_return(env, greedy_actions: np.ndarray, n_eval_episodes: int, max_steps: int = 500):
    """Evaluate mean return and length for a greedy policy.

    Parameters
    ----------
    env : gym.Env
        Environment instance.
    greedy_actions : np.ndarray
        Greedy action index for each state.
    n_eval_episodes : int
        Number of evaluation episodes.
    max_steps : int, default=500
        Maximum number of steps per episode.

    Returns
    -------
    dict[str, float]
        Mean return, return standard deviation, and mean episode length.
    """
    returns = []
    lengths = []

    for _ in range(n_eval_episodes):
        obs, _ = env.reset(seed=int(GLOBAL_RNG.integers(0, 10_000_000)))
        ep_ret = 0.0
        ep_len = 0

        for _ in range(max_steps):
            a = int(greedy_actions[int(obs)])
            obs, r, terminated, truncated, _ = env.step(a)
            ep_ret += float(r)
            ep_len += 1
            if terminated or truncated:
                break

        returns.append(ep_ret)
        lengths.append(ep_len)

    return {
        'mean_return': float(np.mean(returns)),
        'std_return': float(np.std(returns)),
        'mean_length': float(np.mean(lengths)),
    }


### On-policy vs off-policy: running return convergence

This comparison looks at running episode returns instead of only the start-state value estimate.

- **On-policy MC**: sample directly from the deterministic greedy target policy and average the observed returns.
- **Off-policy behavior**: sample from an $\epsilon$-greedy behavior policy and average the raw observed returns.
- **Off-policy MC (weighted IS)**: use the same off-policy episodes, but reweight them to estimate the deterministic greedy target policy.

This makes it easier to compare what the agent actually experiences during sampling versus what importance sampling says about the underlying greedy target policy.


In [ ]:
target_actions_cmp = res_slip['greedy_actions']
n_cmp =  3000
target_policy_cmp = make_policy_from_greedy_actions(target_actions_cmp, env_slip.action_space.n, epsilon=0.0)
behavior_policy_cmp = make_policy_from_greedy_actions(target_actions_cmp, env_slip.action_space.n, epsilon=0.20)

on_returns, on_return_curve = running_mean_episode_returns(
    env=env_slip,
    policy_fn=target_policy_cmp,
    n_episodes=n_cmp,
)

off_behavior_returns, off_behavior_curve = running_mean_episode_returns(
    env=env_slip,
    policy_fn=behavior_policy_cmp,
    n_episodes=n_cmp,
)

off_cmp = off_policy_mc_prediction_start_state(
    env=env_slip,
    greedy_actions=target_actions_cmp,
    n_episodes=n_cmp,
    epsilon_b=0.20,
    gamma=0.99,
)

x = np.arange(1, n_cmp + 1)
fig = plt.figure(figsize=(7.0, 4.2))
plt.plot(x, on_return_curve, label='On-policy sampled return', linewidth=2.0)
plt.plot(x, off_behavior_curve, label='Off-policy behavior return', alpha=0.85)
plt.plot(x, off_cmp['weighted'], label='Off-policy weighted IS target estimate', alpha=0.85)
plt.title('Running mean return on slippery FrozenLake')
plt.xlabel('Episode index')
plt.ylabel('Running mean episodic return')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig.savefig(CELL_OUTPUT_DIR / 'on_vs_off_policy_return_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

print(f"Final on-policy sampled return:   {on_return_curve[-1]:.4f}")
print(f"Final off-policy behavior return: {off_behavior_curve[-1]:.4f}")
print(f"Final weighted IS estimate:       {off_cmp['weighted'][-1]:.4f}")
print(f"Non-zero ratio episodes:          {100.0 * off_cmp['nonzero_ratio_frac']:.2f}%")


The **on-policy sampled return** is the cleanest reference curve. It shows what happens when we actually execute the deterministic greedy target policy and average the observed returns. Since the data is generated by the same policy we want to evaluate, this is the most direct estimate.

The **off-policy behavior return** is usually lower. This is because the behavior policy is $\epsilon$-greedy, so it still takes random exploratory actions. Those extra random actions make the trajectories less efficient and reduce the average return, especially in a noisy environment like slippery FrozenLake.

The **off-policy weighted importance sampling curve** is the key comparison. It uses those same exploratory behavior-policy episodes, but reweights them to estimate what the greedy target policy is worth. This curve should usually lie closer to the on-policy target estimate than to the raw behavior return.

The printed values below the plot help interpret this directly:

- **Final on-policy sampled return**:
  what the target policy achieves when run directly,
- **Final off-policy behavior return**:
  what the exploratory policy actually experiences,
- **Final weighted IS estimate**:
  what off-policy Monte Carlo infers about the target policy from exploratory data,
- **Non-zero ratio episodes**:
  how much of the sampled data remained useful after importance-weighting.

### Another environment: CliffWalking

We can reuse the same on-policy Monte Carlo control loop on a different environment. CliffWalking is harsher than FrozenLake:

- every step gives a negative reward,
- stepping into the cliff gives a large penalty,
- shorter and safer paths produce better returns.

This gives us a second example of the same Monte Carlo control idea, now in a harsher environment.


In [ ]:
env_cliff = gym.make('CliffWalking-v0')

cliff_res = mc_on_policy_control(
    env=env_cliff,
    n_episodes=20_000,
    gamma=
    0.99,
    epsilon=0.2,
    max_steps=100,
)

cliff_learned = evaluate_greedy_return(
    env_cliff,
    cliff_res['greedy_actions'],
    n_eval_episodes=200,
    max_steps=100,
)

print('Learned MC policy:', cliff_learned)


In [ ]:
cliff_gif = s3u.export_greedy_episode_gif(
    env_id='CliffWalking-v0',
    greedy_actions=cliff_res['greedy_actions'],
    output_dir=CELL_OUTPUT_DIR / 'cliffwalking',
    gif_name='cliffwalking_learned_policy.gif',
    max_steps=30,
    fps=3,
)

print('Learned-policy GIF saved to:', cliff_gif['gif_path'])
print(f"Learned-policy episode return: {cliff_gif['episode_return']:.1f}")


The GIF below shows one rollout of the learned Monte Carlo policy on CliffWalking.

<div style="display:flex; justify-content:center; margin:10px 0;">
  <img src="https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/sessions/3_monte_carlo_methods/assets/cell_outputs/cliffwalking/cliffwalking_learned_policy.gif" width="720" />
</div>

This gives a compact showcase of how the same Monte Carlo control loop behaves in a tougher environment than FrozenLake.
